# **SHA-256**

Implementing the SHA-256 in Python from scratch using only the native language components (no imports)

## Functions
These are the basic functions used in the SHA-256 algorithm

In [1]:
def shr(value:int, amount:int) -> int:
    """
    See FIPS 180-4, section 3.2.5, for details on the right shift operation.

    SHR(x, n) = x >> n

    Args:
        value(int): the value to be shifted right.
        amount(int): the number of bits to shift the value by.

    Returns:
        (int): the value shifted right by the given amount.
    """
    return value >> amount


def rotr(value:int, amount:int) -> int:
    """
    See FIPS 180-4, section 3.2.4, for details on the right rotation operation.

    ROTR(x, n) = (x >> n) v (x << w - n)
    Where
        w   =   the word size (32 bits for SHA-256)
        v   =   is the bitwise OR operation.

    The expression above is only a mathematical trick to obtain the right
    rotation of a 32-bit value. The resulting number is padded with 0's on the
    left side due to `value << (32 - amount)`. The bitwise OR operation then
    combines the two shifted values.

    0xFFFFFFFF is a 32-bit value of all 1's. Using the bitwise AND operation
    with 0xFFFFFFFF ensures that the result is a 32-bit value, effectively
    discarding any bits beyond the 32nd bit.

    Args:
        value(int): the value to be rotated right.
        amount(int): the number of bits to rotate the value by.

    Returns:
        (int): the 32-bit value rotated right by the given amount.
    """
    return (value >> amount | value << (32 - amount)) & 0xFFFFFFFF


choose = lambda x, y, z: (x & y) ^ (~x & z)
majority = lambda x, y, z: (x & y) ^ (x & z) ^ (y & z)

Sigma0 = lambda x: rotr(x, 2) ^ rotr(x, 13) ^ rotr(x, 22)
Sigma1 = lambda x: rotr(x, 6) ^ rotr(x, 11) ^ rotr(x, 25)
sigma0 = lambda x: rotr(x, 7) ^ rotr(x, 18) ^ shr(x, 3)
sigma1 = lambda x: rotr(x, 17) ^ rotr(x, 19) ^ shr(x, 10)

## Constants 
The constants defined by the official specification document for SHA-256

In [2]:
"""
SHA-256 Initial Hash Values H, as defined in FIPS 180-4 section 5.3.2.
The first thirty-two bits of the fractional parts of the square roots of the
first eight prime numbers.
"""
H0 = 0x6a09e667
H1 = 0xbb67ae85
H2 = 0x3c6ef372
H3 = 0xa54ff53a
H4 = 0x510e527f
H5 = 0x9b05688c
H6 = 0x1f83d9ab
H7 = 0x5be0cd19

INITIAL_HASH = [H0, H1, H2, H3, H4, H5, H6, H7]


"""
SHA-256 Constants K, as defined in FIPS 180-4 section 4.2.2.
The first 32 bits of the fractional parts of the cube roots of the first 64
prime numbers.
"""
K = [
    0x428a2f98, 0x71374491, 0xb5c0fbcf, 0xe9b5dba5,
    0x3956c25b, 0x59f111f1, 0x923f82a4, 0xab1c5ed5,
    0xd807aa98, 0x12835b01, 0x243185be, 0x550c7dc3,
    0x72be5d74, 0x80deb1fe, 0x9bdc06a7, 0xc19bf174,
    0xe49b69c1, 0xefbe4786, 0x0fc19dc6, 0x240ca1cc,
    0x2de92c6f, 0x4a7484aa, 0x5cb0a9dc, 0x76f988da,
    0x983e5152, 0xa831c66d, 0xb00327c8, 0xbf597fc7,
    0xc6e00bf3, 0xd5a79147, 0x06ca6351, 0x14292967,
    0x27b70a85, 0x2e1b2138, 0x4d2c6dfc, 0x53380d13,
    0x650a7354, 0x766a0abb, 0x81c2c92e, 0x92722c85,
    0xa2bfe8a1, 0xa81a664b, 0xc24b8b70, 0xc76c51a3,
    0xd192e819, 0xd6990624, 0xf40e3585, 0x106aa070,
    0x19a4c116, 0x1e376c08, 0x2748774c, 0x34b0bcb5,
    0x391c0cb3, 0x4ed8aa4a, 0x5b9cca4f, 0x682e6ff3,
    0x748f82ee, 0x78a5636f, 0x84c87814, 0x8cc70208,
    0x90befffa, 0xa4506ceb, 0xbef9a3f7, 0xc67178f2,
]

## Pre-Processing

In [3]:
def pre_processing(input_string: str) -> list[list[int]]:
    """
    Prepare an arbitrary input string for the SHA-256 hashing algorithm.

    The following steps are taken (following the official specification as
    defined in FIPS 180-4, section 5 & section 6.2.1):
        1) The string is encoded to bytes using the UTF-8 format.
        2) The message is padded with the relevant bytes
            - seperator byte
            - empty "zero" bytes
            - length bytes
        3) The output is grouped
            - the message into 512-bit (64-byte) blocks
            - the blocks into 32-bit (4-byte) words
            - the bytes of the words are cast to int

    The reason the words are cast to int, is because Python works nicest with
    integers when performing the arithmatic required for the SHA-256 algorithm
    (e.g. sigma0, sigma1, etc.).

    Args:
        input_string(str): the input string to be prepared for hashing.

    Returns:
        blocks(list[list[int]]): the padded message grouped into 512-bit
            blocks, where each block is a list of 16 32-bit integer words.
    """
    # --- Step 1: Encode --- #
    message = input_string.encode("utf-8")

    # --- Step 2: Padding --- #
    padded = bytearray(message)
    padded.append(0x80)  # seperator byte 0x80 (i.e. 0b10000000)

    # add empty padding byte
    while len(padded) % 64 != 56:
        padded.append(0x00)

    # last 8 bytes reserved for length of original message as big-endian bytes
    padded += (len(message) * 8).to_bytes(8, "big")

    # --- Step 3: Grouping --- #
    blocks = []
    for i in range(0, len(padded), 64):
        block = padded[i:i + 64]
        # each block grouped as 16 4-byte words cast to int
        words = [
            int.from_bytes(block[j:j + 4], "big") & 0xFFFFFFFF
            for j in range(0, 64, 4)
        ]
        blocks.append(words)

    return blocks

# The Algorithm
This is the core logic of the SHA-256 algorithm

In [4]:
def create_message_schedule(block:list[int]):
    """
    Create the message schedule according to step 1 of FIPS 180-4 section 6.2.2

    Args:
        block(list[int]): a single block of 16 32-bit integer words.

    Returns:
        w(list[int]): the message schedule of 64 32-bit integer words.
    """
    # block = 16 words (int)
    w = [word for word in block]
    for t in range(16, 64):
        word = (sigma1(w[t-2]) + w[t-7] + sigma0(w[t-15]) + w[t-16]) & 0xFFFFFFFF
        w.append(word)
    return w


def process_block(block:list[int], hash_values:list[int]):
    """
    Process a single block by applying the hashing algorithm to it according to
    FIPS 180-4 section 6.2.2.

    Args:
        block(list[int]): a single block of 16 32-bit integer words.
        hash_values(list[int]): the current eight 32-bit hash values.

    Returns:
        hash_values(list[int]): the eight 32-bit hash values updated with the
            given block.
    """
    w = create_message_schedule(block)
    a,b,c,d,e,f,g,h = hash_values
    for t in range(64):
        T1 = (h + Sigma1(e) + choose(e, f, g) + K[t] + w[t]) & 0xFFFFFFFF
        T2 = (Sigma0(a) + majority(a, b, c)) & 0xFFFFFFFF
        h = g
        g = f
        f = e
        e = (d + T1) & 0xFFFFFFFF
        d = c
        c = b
        b = a
        a = (T1 + T2) & 0xFFFFFFFF

    # update hash values (but keep them as 32-bit values)
    hash_values[0] = (hash_values[0] + a) & 0xFFFFFFFF
    hash_values[1] = (hash_values[1] + b) & 0xFFFFFFFF
    hash_values[2] = (hash_values[2] + c) & 0xFFFFFFFF
    hash_values[3] = (hash_values[3] + d) & 0xFFFFFFFF
    hash_values[4] = (hash_values[4] + e) & 0xFFFFFFFF
    hash_values[5] = (hash_values[5] + f) & 0xFFFFFFFF
    hash_values[6] = (hash_values[6] + g) & 0xFFFFFFFF
    hash_values[7] = (hash_values[7] + h) & 0xFFFFFFFF
    return hash_values

def process_blocks(blocks:list[list[int]]) -> list[int]:
    """
    Given a list of blocks, initialize the initial hash values, and updated them
    by running each block through the hashing algorithm (see `process_block`
    function). See FIPS 180-4 section 6.2.2 for more detail.

    Args:
        blocks(list[list[int]]): the list of blocks, where each block is a list
            of 16 32-bit integer words.

    Returns:
        hash_values(list[int]): the final eight 32-bit hash values after
            processing all blocks.
    """
    hash_values = [h for h in INITIAL_HASH]
    for block in blocks:
        hash_values = process_block(block, hash_values)
    return hash_values

def finalize_hash_output(hash_values:list[int]) -> str:
    """
    Given a set of hash values, convert them to a hex string.

    Each hash value represents a 32-bit word, which is equivalent to 4 bytes.
    Each value is first converted to a bytes object (`v.to_bytes(4, "big")`),
    and then all 8 hash values as bytes are concatenated into a single bytes
    object. Finally, the bytes object is returned as a hex string.

    Args:
        hash_values(list[int]): the eight final 32-bit hash values.

    Returns:
        (str): the concatenated hash values as a hexadecimal string.
    """
    output = b"".join(v.to_bytes(4, "big") for v in hash_values)
    return output.hex()

In [5]:
def sha256(input_string:str)->str:
    """
    Compute the SHA-256 hash of an input string.

    Args:
        input_string(str): the input string to be hashed.

    Returns:
        (str): the SHA-256 hash of the input string as a hexadecimal string.
    """
    blocks = pre_processing(input_string)
    hash_values = process_blocks(blocks)
    return finalize_hash_output(hash_values)

## Example Outputs

In [6]:
sha256('abc')

'ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad'

In [7]:
sha256('hello world')

'b94d27b9934d3e08a52e52d7da7dabfac484efe37a5380ee9088f7ace2efcde9'

In [8]:
sha256('Hello there')

'4e47826698bb4630fb4451010062fadbf85d61427cbdfaed7ad0f23f239bed89'

In [9]:
sha256('May the Force be with you')

'adf76935714ead08f9cb04c2c8f41b56f417aeab982ccbf0d29922f3da366145'

In [10]:
sha256("""Never gonna give you up
Never gonna let you down
Never gonna run around and desert you
Never gonna make you cry
Never gonna say goodbye
Never gonna tell a lie and hurt you""")

'447c444bf657877bd1a035a38385f271820bb95be8cbc5b8678667d219c19e36'